# Hands-On Lab: Gradient Boosting & XGBoost
Accompanies **Chapter 16: Gradient Boosting Machines** and **Chapter 17: Extreme Gradient Boosting (XGBoost)**.

In this lab, we:
1. Train a Gradient Boosting classifier and track staged deviance across boosting rounds.
2. Train an XGBoost model using the official `xgboost` library.
3. Demonstrate early stopping to prevent sequential overfitting.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X = df[numeric_cols].fillna(df[numeric_cols].median())
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

train_loss = [log_loss(y_train, p) for p in gbm.staged_predict_proba(X_train)]
test_loss = [log_loss(y_test, p) for p in gbm.staged_predict_proba(X_test)]

plt.figure(figsize=(9, 5))
plt.plot(train_loss, label='Train Log-Loss', color='blue', linewidth=2)
plt.plot(test_loss, label='Test Log-Loss', color='red', linewidth=2)
plt.xlabel('Boosting Iterations', fontsize=11)
plt.ylabel('Negative Log-Likelihood', fontsize=11)
plt.title('GBM Staged Loss Trajectory', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()